# RecoMart Synthetic Data Generator
## Timestamped Data Generation for E-commerce Analytics

This notebook generates realistic synthetic e-commerce data and writes it to the volume with timestamps for version control.

**Output Location:** `/Volumes/workspace/recomart_source_data/raw/data/`

**File Naming Pattern:** `{filename}_{YYYYMMDD_HHMMSS}.{ext}`

**Data Types:**
* User Interaction Logs (CSV)
* Purchase History (CSV)
* Product Catalog (JSON)
* External Signals (JSON)

**Features:**
* Realistic data distributions
* Temporal patterns
* Version management (keeps last 10 versions)
* Production-ready error handling

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# Configuration & Imports
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import json
import random
from datetime import datetime, timedelta
import uuid
from pathlib import Path

# ── CONFIGURATION ─────────────────────────────────────────────────────────────
OUTPUT_BASE_PATH = "/Volumes/workspace/recomart_source_data/raw/data"

# Data generation parameters
NUM_USERS = 1000
NUM_ITEMS = 500
NUM_EVENTS = 5000
NUM_PURCHASES = 1000

# Keep only the last N versions of each file type
MAX_FILE_VERSIONS = 10

# Seed for reproducibility (optional - comment out for random data each run)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

print("✓ Configuration loaded")
print(f"  Output path: {OUTPUT_BASE_PATH}")
print(f"  Data size: {NUM_USERS} users, {NUM_ITEMS} items, {NUM_EVENTS} events, {NUM_PURCHASES} purchases")

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# Helper Functions
# ══════════════════════════════════════════════════════════════════════════════

def get_timestamp_suffix():
    """Generate timestamp suffix in YYYYMMDD_HHMMSS format"""
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def write_with_timestamp(data, base_path: str, file_prefix: str, file_format: str, timeout: int = 30):
    """
    Write data to volume with timestamp suffix (serverless compatible).
    Simplified version without automatic cleanup to avoid hanging.
    
    Args:
        data: pandas DataFrame or dict (for JSON)
        base_path: Base directory path
        file_prefix: Prefix for filename
        file_format: File format ('csv' or 'json')
        timeout: Timeout in seconds (default 30)
    
    Returns:
        str: Full path of written file
    """
    import signal
    
    timestamp = get_timestamp_suffix()
    filename = f"{file_prefix}_{timestamp}.{file_format}"
    file_path = f"{base_path}/{filename}"
    
    try:
        # Handle dict (JSON) input
        if isinstance(data, dict):
            import json
            json_str = json.dumps(data, indent=2)
            dbutils.fs.put(file_path, json_str, overwrite=True)
            record_count = data.get('count', len(data.get('data', [])))
        else:
            # For CSV: Use pandas to_csv with StringIO, then write with dbutils
            from io import StringIO
            csv_buffer = StringIO()
            data.to_csv(csv_buffer, index=False)
            csv_content = csv_buffer.getvalue()
            dbutils.fs.put(file_path, csv_content, overwrite=True)
            record_count = len(data)
        
        print(f"✓ Written: {filename} ({record_count} records)")
        return file_path
        
    except Exception as e:
        print(f"❌ Error writing {filename}: {str(e)}")
        raise

def cleanup_old_versions_manual(base_path: str, file_prefix: str, extension: str, keep_last: int = 10):
    """
    Manually cleanup old versions (call this separately if needed).
    Separated from write function to avoid blocking writes.
    
    Args:
        base_path: Directory containing the files
        file_prefix: File name prefix (e.g., 'purchase_history')
        extension: File extension (e.g., 'csv', 'json')
        keep_last: Number of versions to keep (default 10)
    """
    try:
        print(f"\n🧹 Cleaning up old {file_prefix}.{extension} files...")
        # List all files matching pattern
        files = dbutils.fs.ls(base_path)
        matching_files = [
            f for f in files 
            if f.name.startswith(file_prefix) and f.name.endswith(f'.{extension}')
        ]
        
        print(f"  Found {len(matching_files)} {file_prefix} files")
        
        if len(matching_files) > keep_last:
            # Sort by modification time (oldest first)
            matching_files.sort(key=lambda x: x.modificationTime)
            
            # Delete oldest files
            files_to_delete = matching_files[:-keep_last]
            for f in files_to_delete:
                dbutils.fs.rm(f.path)
                print(f"  Deleted: {f.name}")
            
            print(f"  ✓ Kept last {keep_last} versions")
        else:
            print(f"  ✓ No cleanup needed (only {len(matching_files)} files)")
                
    except Exception as e:
        print(f"  ⚠️  Cleanup failed: {e}")

print("✓ Helper functions defined")

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# Generate Purchase History (CSV)
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("Generating Purchase History...")
print("="*80)

# Payment method distribution
payment_methods = ['credit_card', 'debit_card', 'paypal', 'upi']
payment_weights = [0.40, 0.30, 0.20, 0.10]

# Generate purchase transactions
purchase_data = []
for _ in range(NUM_PURCHASES):
    transaction_id = f"TXN{uuid.uuid4().hex[:10].upper()}"
    user_id = f"U{random.randint(1, NUM_USERS):04d}"
    item_id = f"I{random.randint(1, NUM_ITEMS):04d}"
    quantity = random.choices([1, 2, 3, 4, 5], weights=[0.70, 0.15, 0.08, 0.05, 0.02])[0]
    
    # Price follows a realistic distribution
    base_price = np.random.lognormal(mean=3.5, sigma=0.8)  # Log-normal for realistic price distribution
    amount = round(base_price * quantity, 2)
    
    payment_method = random.choices(payment_methods, weights=payment_weights)[0]
    
    # Timestamp within last 30 days
    days_ago = random.randint(0, 30)
    hours_ago = random.randint(0, 23)
    timestamp = datetime.now() - timedelta(days=days_ago, hours=hours_ago)
    
    purchase_data.append({
        'transaction_id': transaction_id,
        'user_id': user_id,
        'item_id': item_id,
        'quantity': quantity,
        'amount': amount,
        'payment_method': payment_method,
        'timestamp': timestamp.strftime('%Y-%m-%d %H:%M:%S')
    })

df_purchases = pd.DataFrame(purchase_data)

# Sort by timestamp
df_purchases = df_purchases.sort_values('timestamp').reset_index(drop=True)

print(f"✓ Generated {len(df_purchases):,} purchase records")
print(f"  Total revenue: ${df_purchases['amount'].sum():,.2f}")
print(f"  Average transaction: ${df_purchases['amount'].mean():.2f}")
print(f"  Payment method distribution:")
for method in payment_methods:
    count = len(df_purchases[df_purchases['payment_method'] == method])
    pct = count / len(df_purchases) * 100
    print(f"    {method:15s}: {count:5d} ({pct:5.1f}%)")

# Write to volume with timestamp
write_with_timestamp(df_purchases, OUTPUT_BASE_PATH, 'purchase_history', 'csv')

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# Generate Product Catalog (JSON)
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("Generating Product Catalog...")
print("="*80)

# Categories and their typical products
categories_data = {
    'Electronics': ['Smartphone', 'Laptop', 'Headphones', 'Tablet', 'Smartwatch', 'Camera', 'Speaker'],
    'Clothing': ['T-Shirt', 'Jeans', 'Dress', 'Jacket', 'Shoes', 'Sweater', 'Hoodie'],
    'Home & Garden': ['Lamp', 'Chair', 'Table', 'Plant', 'Cushion', 'Rug', 'Mirror'],
    'Books': ['Novel', 'Biography', 'Textbook', 'Magazine', 'Comic', 'Journal', 'Guide'],
    'Sports': ['Yoga Mat', 'Dumbbells', 'Running Shoes', 'Bicycle', 'Tennis Racket', 'Ball', 'Resistance Band'],
    'Beauty': ['Lipstick', 'Moisturizer', 'Perfume', 'Shampoo', 'Facial Mask', 'Foundation', 'Serum']
}

brands = ['BrandA', 'BrandB', 'BrandC', 'BrandD', 'BrandE', 'BrandF', 'BrandG', 'BrandH']

# Generate product catalog
product_catalog = []
for i in range(1, NUM_ITEMS + 1):
    item_id = f"I{i:04d}"
    category = random.choice(list(categories_data.keys()))
    product_type = random.choice(categories_data[category])
    name = f"{product_type} - {random.choice(brands)}"
    
    # Price distribution varies by category
    if category == 'Electronics':
        price = round(random.uniform(50, 2000), 2)
    elif category == 'Clothing':
        price = round(random.uniform(15, 200), 2)
    elif category in ['Books', 'Beauty']:
        price = round(random.uniform(10, 100), 2)
    else:
        price = round(random.uniform(20, 500), 2)
    
    brand = random.choice(brands)
    rating = round(random.uniform(2.5, 5.0), 1)
    stock = random.randint(0, 500)
    description = f"High-quality {product_type.lower()} from {brand}. {category} category."
    
    product_catalog.append({
        'item_id': item_id,
        'name': name,
        'category': category,
        'price': price,
        'brand': brand,
        'rating': rating,
        'stock': stock,
        'description': description
    })

# Wrap in API-style response format
catalog_json = {
    'status': 'success',
    'timestamp': datetime.now().isoformat(),
    'count': len(product_catalog),
    'data': product_catalog
}

print(f"✓ Generated {len(product_catalog):,} product records")
print(f"  Category distribution:")
for category in categories_data.keys():
    count = sum(1 for p in product_catalog if p['category'] == category)
    pct = count / len(product_catalog) * 100
    print(f"    {category:15s}: {count:5d} ({pct:5.1f}%)")

# Write to volume with timestamp
write_with_timestamp(catalog_json, OUTPUT_BASE_PATH, 'product_catalog', 'json')

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# Generate External Signals (JSON)
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("Generating External Signals...")
print("="*80)

# Generate external trend signals for items
external_signals = []
for i in range(1, NUM_ITEMS + 1):
    signal_id = f"SIG{uuid.uuid4().hex[:8].upper()}"
    item_id = f"I{i:04d}"
    
    # Trend score (0-100, higher means trending)
    trend_score = round(random.uniform(0, 100), 2)
    
    # Social mentions (realistic distribution - most items have few mentions)
    if random.random() < 0.1:  # 10% are viral
        social_mentions = random.randint(1000, 50000)
    elif random.random() < 0.3:  # 30% are popular
        social_mentions = random.randint(100, 1000)
    else:  # 60% have low mentions
        social_mentions = random.randint(0, 100)
    
    timestamp = datetime.now().isoformat()
    
    external_signals.append({
        'signal_id': signal_id,
        'item_id': item_id,
        'trend_score': trend_score,
        'social_mentions': social_mentions,
        'timestamp': timestamp
    })

# Wrap in API-style response format
signals_json = {
    'status': 'success',
    'timestamp': datetime.now().isoformat(),
    'count': len(external_signals),
    'data': external_signals
}

print(f"✓ Generated {len(external_signals):,} external signal records")
print(f"  Average trend score: {sum(s['trend_score'] for s in external_signals) / len(external_signals):.2f}")
print(f"  Total social mentions: {sum(s['social_mentions'] for s in external_signals):,}")

# Mention distribution
viral_count = sum(1 for s in external_signals if s['social_mentions'] >= 1000)
popular_count = sum(1 for s in external_signals if 100 <= s['social_mentions'] < 1000)
low_count = sum(1 for s in external_signals if s['social_mentions'] < 100)

print(f"  Social mention distribution:")
print(f"    Viral (≥1000):     {viral_count:5d} ({viral_count/len(external_signals)*100:5.1f}%)")
print(f"    Popular (100-999): {popular_count:5d} ({popular_count/len(external_signals)*100:5.1f}%)")
print(f"    Low (<100):        {low_count:5d} ({low_count/len(external_signals)*100:5.1f}%)")

# Write to volume with timestamp
write_with_timestamp(signals_json, OUTPUT_BASE_PATH, 'external_signals', 'json')

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# Summary & Verification (BATCH ONLY)
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("BATCH DATA GENERATION COMPLETE")
print("="*80)

print(f"\n📊 Summary:")
print(f"  ✓ Purchase History:       {len(df_purchases):,} records")
print(f"  ✓ Product Catalog:        {len(product_catalog):,} records")
print(f"  ✓ External Signals:       {len(external_signals):,} records")
print(f"  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"  📁 Total records:         {len(df_purchases) + len(product_catalog) + len(external_signals):,}")

print(f"\n📂 Output Location: {OUTPUT_BASE_PATH}")
print(f"\n⚠️  NOTE: User interaction logs are generated by STREAMING pipeline")
print(f"           See: task1_streaming_data_generator")

# List generated files
print(f"\n📋 Generated Files:")
try:
    files = dbutils.fs.ls(OUTPUT_BASE_PATH)
    timestamp = get_timestamp_suffix()
    
    # Group files by type (BATCH ONLY)
    file_types = ['purchase_history', 'product_catalog', 'external_signals']
    
    for file_type in file_types:
        matching = [f for f in files if f.name.startswith(file_type)]
        print(f"\n  {file_type}:")
        if matching:
            # Sort by modification time (newest first)
            matching.sort(key=lambda x: x.modificationTime, reverse=True)
            for i, f in enumerate(matching[:5]):  # Show only last 5
                size_kb = f.size / 1024
                mod_time = datetime.fromtimestamp(f.modificationTime / 1000)
                marker = "← LATEST" if i == 0 else ""
                print(f"    • {f.name:60s} ({size_kb:6.1f} KB)  {mod_time.strftime('%Y-%m-%d %H:%M:%S')}  {marker}")
        else:
            print(f"    No files found")
            
except Exception as e:
    print(f"  ⚠️  Could not list files: {e}")

print(f"\n" + "="*80)
print("✅ Ready for batch ingestion! The task2_databricks_ingestion notebook will")
print("   automatically pick up the latest timestamped files.")
print("="*80)